# Conversion of CFDBench Tube/prop dataset to PDEBench INS

Summary:
1) Let channel 3 be geo mask
2) Remove simulations with <20 timesteps
3) For those >20 timesteps, split into segments of 20 and concat accordingly

In [32]:
# Tube/prop

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/tube/prop'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        u = np.load(os.path.join(case_dir, 'u.npy'), mmap_mode='r')
        v = np.load(os.path.join(case_dir, 'v.npy'), mmap_mode='r')

        print(f"{case}: u{u.shape}, v{v.shape}")

case0050: u(14, 64, 64), v(14, 64, 64)
case0051: u(21, 64, 64), v(21, 64, 64)
case0052: u(20, 64, 64), v(20, 64, 64)
case0053: u(20, 64, 64), v(20, 64, 64)
case0054: u(20, 64, 64), v(20, 64, 64)
case0055: u(20, 64, 64), v(20, 64, 64)
case0056: u(21, 64, 64), v(21, 64, 64)
case0057: u(21, 64, 64), v(21, 64, 64)
case0058: u(20, 64, 64), v(20, 64, 64)
case0059: u(20, 64, 64), v(20, 64, 64)
case0060: u(26, 64, 64), v(26, 64, 64)
case0061: u(19, 64, 64), v(19, 64, 64)
case0062: u(18, 64, 64), v(18, 64, 64)
case0063: u(19, 64, 64), v(19, 64, 64)
case0064: u(13, 64, 64), v(13, 64, 64)
case0065: u(19, 64, 64), v(19, 64, 64)
case0066: u(19, 64, 64), v(19, 64, 64)
case0067: u(19, 64, 64), v(19, 64, 64)
case0068: u(19, 64, 64), v(19, 64, 64)
case0069: u(20, 64, 64), v(20, 64, 64)
case0070: u(27, 64, 64), v(27, 64, 64)
case0071: u(19, 64, 64), v(19, 64, 64)
case0072: u(18, 64, 64), v(18, 64, 64)
case0073: u(19, 64, 64), v(19, 64, 64)
case0074: u(19, 64, 64), v(19, 64, 64)
case0075: u(19, 64, 64), 

# Padding of Dataset
Based on tube.py:

```python
def load_case_data(case_dir: Path) -> Tuple[np.ndarray, Dict[str, float]]:
    """
    Load from the file that I have preprocessed, and pad the boundary
    conditions, turn into a numpy array of features.

    The shape of both u and v is (time steps, height, width)
    """
    case_params = load_json(case_dir / "case.json")
    # print(case_params)

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)
    # Shape of u and v: (time steps, height, width)

    # Mask
    mask = np.ones_like(u)

    # Pad the left side
    u = np.pad(
        u,
        ((0, 0), (0, 0), (1, 0)),
        mode="constant",
        constant_values=case_params["vel_in"],
    )
    v = np.pad(v, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    # # Pad the top and bottom
    u = np.pad(u, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    v = np.pad(v, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    # mask = 1 - mask
    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)
    return features, case_params
```

In [33]:
# Padding
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import shutil

def load_json(path):
    with open(path, 'r', encoding='utf8') as f:
        return json.load(f)

input_path = Path('/Volumes/T7/CFDBench/tube/prop')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_prop/tube_prop_padded')

for case_dir in tqdm(sorted(input_path.iterdir())):

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)

    case_params = load_json(case_dir / "case.json")

    mask = np.ones_like(u)

    # Pad left
    u = np.pad(u, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=case_params["vel_in"],)
    v = np.pad(v, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    # # Pad the top and bottom
    u = np.pad(u, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    v = np.pad(v, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    np.save(case_out_dir / "features.npy", features)
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')


100%|██████████| 100/100 [00:00<00:00, 120.60it/s]

Done


# Sanity Check

In [36]:
# Padded check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/tube_prop/tube_prop_padded'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0050: features(14, 3, 66, 65)
case0051: features(21, 3, 66, 65)
case0052: features(20, 3, 66, 65)
case0053: features(20, 3, 66, 65)
case0054: features(20, 3, 66, 65)
case0055: features(20, 3, 66, 65)
case0056: features(21, 3, 66, 65)
case0057: features(21, 3, 66, 65)
case0058: features(20, 3, 66, 65)
case0059: features(20, 3, 66, 65)
case0060: features(26, 3, 66, 65)
case0061: features(19, 3, 66, 65)
case0062: features(18, 3, 66, 65)
case0063: features(19, 3, 66, 65)
case0064: features(13, 3, 66, 65)
case0065: features(19, 3, 66, 65)
case0066: features(19, 3, 66, 65)
case0067: features(19, 3, 66, 65)
case0068: features(19, 3, 66, 65)
case0069: features(20, 3, 66, 65)
case0070: features(27, 3, 66, 65)
case0071: features(19, 3, 66, 65)
case0072: features(18, 3, 66, 65)
case0073: features(19, 3, 66, 65)
case0074: features(19, 3, 66, 65)
case0075: features(19, 3, 66, 65)
case0076: features(18, 3, 66, 65)
case0077: features(19, 3, 66, 65)
case0078: features(18, 3, 66, 65)
case0079: feat

# Convergence check
Based on tube.py/TubeFlowAutoDataset (Lines 245-261):

```python
for i in range(num_steps):
                inp = torch.tensor(inputs[i], dtype=torch.float32)  # (2, h, w)
                out = torch.tensor(outputs[i], dtype=torch.float32)

                # Check for convergence
                inp_magn = torch.sqrt(inp[0] ** 2 + inp[1] ** 2)
                out_magn = torch.sqrt(out[0] ** 2 + out[1] ** 2)
                diff = torch.abs(inp_magn - out_magn).mean()
                # print(f"Mean difference: {diff}")
                if diff < self.stable_state_diff:
                    print(f"Converged at {i} out of {num_steps}, {this_case_params}")
                    break
                assert not torch.isnan(inp).any()
                assert not torch.isnan(out).any()
                all_inputs.append(inp)
                all_labels.append(out)
                all_case_ids.append(case_id)
```

In [39]:
# Convergence
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_prop/tube_prop_padded')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_prop/tube_prop_convergence')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    converged_idx = num_timesteps - 1

    for i in range(converged_idx):
        u_in, v_in = features[i, 0, :, :], features[i, 1, :, :]
        u_out, v_out = features[i+1, 0, :, :], features[i+1, 1, :, :]

        mag_in = np.sqrt(u_in**2 + v_in**2)
        mag_out = np.sqrt(u_out**2 + v_out**2)

        diff = np.abs(mag_in - mag_out).mean()

        if diff < 1e-3:
            converged_idx = i
            break

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    truncated_features = features[:converged_idx + 1]
    np.save(case_out_dir / "features.npy", truncated_features.astype('float32'))
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')

100%|██████████| 101/101 [00:00<00:00, 129.97it/s]

Done


# Sanity Check

In [41]:
# Convergence check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/tube_prop/tube_prop_convergence'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0050: features(14, 3, 66, 65)
case0051: features(17, 3, 66, 65)
case0052: features(17, 3, 66, 65)
case0053: features(17, 3, 66, 65)
case0054: features(17, 3, 66, 65)
case0055: features(17, 3, 66, 65)
case0056: features(17, 3, 66, 65)
case0057: features(18, 3, 66, 65)
case0058: features(18, 3, 66, 65)
case0059: features(18, 3, 66, 65)
case0060: features(20, 3, 66, 65)
case0061: features(17, 3, 66, 65)
case0062: features(16, 3, 66, 65)
case0063: features(16, 3, 66, 65)
case0064: features(13, 3, 66, 65)
case0065: features(17, 3, 66, 65)
case0066: features(17, 3, 66, 65)
case0067: features(17, 3, 66, 65)
case0068: features(17, 3, 66, 65)
case0069: features(17, 3, 66, 65)
case0070: features(21, 3, 66, 65)
case0071: features(17, 3, 66, 65)
case0072: features(16, 3, 66, 65)
case0073: features(16, 3, 66, 65)
case0074: features(16, 3, 66, 65)
case0075: features(16, 3, 66, 65)
case0076: features(16, 3, 66, 65)
case0077: features(16, 3, 66, 65)
case0078: features(16, 3, 66, 65)
case0079: feat

# Segmentation

Splits each truncated sequence after convergence step above into fixed length of 20 timesteps, discarding the remainder.

Based on convert_cfdbench.py:

```python
def split_trajectory(data_list, time_step, grid_size=64):
    traj_split = []
    for i, x in enumerate(data_list):
        T = x.shape[0]
        # num_segments = int(np.ceil(T / time_step))
        num_segments = int(np.floor(T / time_step))
        if num_segments < 1:
            continue

        padded_length = num_segments * time_step
        padded_array = x[:padded_length]
        # padded_array = np.zeros((padded_length, *x.shape[1:]))

        # Copy the original data into the padded array
        # padded_array[:T, ...] = x

        # # If needed, pad the last segment with the last frame of the original array
        # if T % time_step != 0:
        #     last_frame = x[-1, ...]
        #     padded_array[T:, ...] = last_frame

        # Reshape the array into segments
        padded_array = F.interpolate(
            torch.from_numpy(padded_array).float(), size=(grid_size, grid_size), mode="bilinear", align_corners=True
        ).numpy()
        padded_array = padded_array.reshape((num_segments, time_step, *padded_array.shape[1:]))

        traj_split.append(padded_array)

    traj_split = np.concatenate(traj_split, axis=0)
    return traj_split
```

In [40]:
# Segmentation
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_prop/tube_prop_convergence')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_prop/tube_prop_segmented')

t_len = 20

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"
    json_file = case_dir / "case.json"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    num_segments = num_timesteps // t_len

    if num_segments == 0:
        continue

    for i in range(num_segments):
        start = i * t_len
        end = start + t_len

        segment = features[start:end]

        segment_name = f"{case_dir.name}_seg{i}"
        segment_out_dir = output_path / segment_name
        segment_out_dir.mkdir(exist_ok=True)

        np.save(segment_out_dir / "features.npy", segment.astype('float32'))
        shutil.copy(json_file, segment_out_dir / "case.json")

print('Done')

100%|██████████| 101/101 [00:00<00:00, 350.40it/s]

Done


# Sanity Check

In [1]:
# Segmentation check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/tube_prop/tube_prop_segmented'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0060_seg0: features(20, 3, 66, 65)
case0070_seg0: features(20, 3, 66, 65)
case0080_seg0: features(20, 3, 66, 65)
case0090_seg0: features(20, 3, 66, 65)
case0100_seg0: features(20, 3, 66, 65)
case0110_seg0: features(20, 3, 66, 65)
case0120_seg0: features(20, 3, 66, 65)
case0130_seg0: features(20, 3, 66, 65)
case0140_seg0: features(20, 3, 66, 65)


# Convert to hdf5

In [2]:
# Convert to hdf5

import torch
import torch.nn.functional as F
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_prop/tube_prop_segmented')
output = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_prop/tube_prop_final/tube_prop_converted.h5')

t_len = 20
case_folders = sorted([f for f in input_path.iterdir() if f.is_dir()])
num_segments = len(case_folders)

with h5py.File(output, 'w') as f:
    velocity = f.create_dataset('velocity', shape=(num_segments, t_len, 512, 512, 2),
                                dtype='float32', chunks=(1, t_len, 512, 512, 2))
    particles = f.create_dataset('particles', shape=(num_segments, t_len, 512, 512, 1),
                                 dtype='float32', chunks=(1, t_len, 512, 512, 1))

    for i, case_dir in enumerate(tqdm(case_folders)):
        features = np.load(case_dir / 'features.npy')

        u = features[:, 0, :, :]
        v = features[:, 1, :, :]
        mask = features[:, 2, :, :]

        u_t = torch.from_numpy(u).unsqueeze(1)
        v_t = torch.from_numpy(v).unsqueeze(1)
        mask_t = torch.from_numpy(mask).unsqueeze(1)

        # Upsample to 512x512
        u_up = F.interpolate(u_t, size=(512, 512), mode='bilinear', align_corners=True)
        v_up = F.interpolate(v_t, size=(512, 512), mode='bilinear', align_corners=True)
        mask_up = F.interpolate(mask_t, size=(512, 512), mode='nearest')

        # Permute
        velocity_stack = torch.cat([u_up, v_up], dim=1).permute(0, 2, 3, 1).numpy()
        mask_stack = mask_up.permute(0, 2, 3, 1).numpy()

        velocity[i] = velocity_stack
        particles[i] = mask_stack

print('Done')

100%|██████████| 9/9 [00:00<00:00,  9.13it/s]

Done


In [3]:
# HDF5 check

import h5py

output = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_prop/tube_prop_final/tube_prop_converted.h5')

with h5py.File(output, 'r') as f:
    print(f.keys())
    print(f['velocity'])
    print(f['particles'])

<KeysViewHDF5 ['particles', 'velocity']>
<HDF5 dataset "velocity": shape (9, 20, 512, 512, 2), type "<f4">
<HDF5 dataset "particles": shape (9, 20, 512, 512, 1), type "<f4">


# Inference results for converted TUBE/PROP

In [4]:
# TUBE/PROP RESULTS
# INFO - 04/06/26 08:48:36 - 0:01:19 - Evaluation Stats (total size = 9)
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | type         | dim   | size   |   data_loss |   rel l2 |   rel l2 step 1 |   rel l2 step 5 |   rel l2 step 10 |   rel l2 interior |
# +==============+=======+========+=============+==========+=================+=================+==================+===================+
# | incom_ns     | 3     | 9      |    0.751728 |   0.3272 |          0.2590 |          0.3125 |           0.3272 |            0.3257 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | AVE_BY_CLASS | -     | -      |    0.751728 |   0.3272 |          0.2590 |          0.3125 |           0.3272 |            0.3257 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# INFO - 04/06/26 08:48:36 - 0:01:19 - Additional Stats for Rel L2 Error:
#     +----------+--------+--------+--------+--------+--------+----------+
#     | type     |   size |   mean |    std |    min |    max |   median |
#     +==========+========+========+========+========+========+==========+
#     | incom_ns |      9 | 0.3272 | 0.0009 | 0.3261 | 0.3289 |   0.3269 |
#     +----------+--------+--------+--------+--------+--------+----------+
# INFO - 04/06/26 08:48:36 - 0:01:19 - Eval | data loss = 0.751728 | rel l2 = 0.327166 | rel l2 step 1 = 0.259047 | rel l2 step 5 = 0.312472 | rel l2 step 10 = 0.327166 | rel l2 interior = 0.325727
# INFO - 04/06/26 08:48:36 - 0:01:19 -  MEM: 0.00 MB

# Process is repeated for TUBE/GEO -> PDEBench INS

In [5]:
# Tube/geo
# Let channel 3 be geo mask
# Remove simulations with <20 timesteps
# For those >20 timesteps, split into segments of 20 and concat accordingly

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/tube/geo'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        u = np.load(os.path.join(case_dir, 'u.npy'), mmap_mode='r')
        v = np.load(os.path.join(case_dir, 'v.npy'), mmap_mode='r')

        print(f"{case}: u{u.shape}, v{v.shape}")

case0150: u(10, 64, 64), v(10, 64, 64)
case0151: u(12, 64, 64), v(12, 64, 64)
case0152: u(14, 64, 64), v(14, 64, 64)
case0153: u(26, 64, 64), v(26, 64, 64)
case0154: u(29, 64, 64), v(29, 64, 64)
case0155: u(37, 64, 64), v(37, 64, 64)
case0156: u(26, 64, 64), v(26, 64, 64)
case0157: u(30, 64, 64), v(30, 64, 64)
case0158: u(55, 64, 64), v(55, 64, 64)
case0159: u(52, 64, 64), v(52, 64, 64)
case0160: u(23, 64, 64), v(23, 64, 64)
case0161: u(14, 64, 64), v(14, 64, 64)
case0162: u(41, 64, 64), v(41, 64, 64)
case0163: u(31, 64, 64), v(31, 64, 64)
case0164: u(72, 64, 64), v(72, 64, 64)
case0165: u(17, 64, 64), v(17, 64, 64)
case0166: u(21, 64, 64), v(21, 64, 64)
case0167: u(31, 64, 64), v(31, 64, 64)
case0168: u(51, 64, 64), v(51, 64, 64)
case0169: u(64, 64, 64), v(64, 64, 64)
case0170: u(20, 64, 64), v(20, 64, 64)
case0171: u(27, 64, 64), v(27, 64, 64)
case0172: u(45, 64, 64), v(45, 64, 64)
case0173: u(61, 64, 64), v(61, 64, 64)
case0174: u(78, 64, 64), v(78, 64, 64)


In [6]:
# Padding
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import shutil

def load_json(path):
    with open(path, 'r', encoding='utf8') as f:
        return json.load(f)

input_path = Path('/Volumes/T7/CFDBench/tube/geo')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_geo/tube_geo_padded')

for case_dir in tqdm(sorted(input_path.iterdir())):

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)

    case_params = load_json(case_dir / "case.json")

    mask = np.ones_like(u)

    # Pad left
    u = np.pad(u, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=case_params["vel_in"],)
    v = np.pad(v, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    # # Pad the top and bottom
    u = np.pad(u, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    v = np.pad(v, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    np.save(case_out_dir / "features.npy", features)
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')


100%|██████████| 25/25 [00:00<00:00, 83.49it/s]

Done


In [7]:
# Padded check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/tube_geo/tube_geo_padded'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0150: features(10, 3, 66, 65)
case0151: features(12, 3, 66, 65)
case0152: features(14, 3, 66, 65)
case0153: features(26, 3, 66, 65)
case0154: features(29, 3, 66, 65)
case0155: features(37, 3, 66, 65)
case0156: features(26, 3, 66, 65)
case0157: features(30, 3, 66, 65)
case0158: features(55, 3, 66, 65)
case0159: features(52, 3, 66, 65)
case0160: features(23, 3, 66, 65)
case0161: features(14, 3, 66, 65)
case0162: features(41, 3, 66, 65)
case0163: features(31, 3, 66, 65)
case0164: features(72, 3, 66, 65)
case0165: features(17, 3, 66, 65)
case0166: features(21, 3, 66, 65)
case0167: features(31, 3, 66, 65)
case0168: features(51, 3, 66, 65)
case0169: features(64, 3, 66, 65)
case0170: features(20, 3, 66, 65)
case0171: features(27, 3, 66, 65)
case0172: features(45, 3, 66, 65)
case0173: features(61, 3, 66, 65)
case0174: features(78, 3, 66, 65)


In [8]:
# Convergence
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_geo/tube_geo_padded')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_geo/tube_geo_convergence')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    converged_idx = num_timesteps - 1

    for i in range(converged_idx):
        u_in, v_in = features[i, 0, :, :], features[i, 1, :, :]
        u_out, v_out = features[i+1, 0, :, :], features[i+1, 1, :, :]

        mag_in = np.sqrt(u_in**2 + v_in**2)
        mag_out = np.sqrt(u_out**2 + v_out**2)

        diff = np.abs(mag_in - mag_out).mean()

        if diff < 1e-3:
            converged_idx = i
            break

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    truncated_features = features[:converged_idx + 1]
    np.save(case_out_dir / "features.npy", truncated_features.astype('float32'))
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')

100%|██████████| 25/25 [00:00<00:00, 212.86it/s]

Done


In [9]:
# Segmentation
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_geo/tube_geo_convergence')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_geo/tube_geo_segmented')

t_len = 20

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"
    json_file = case_dir / "case.json"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    num_segments = num_timesteps // t_len

    if num_segments == 0:
        continue

    for i in range(num_segments):
        start = i * t_len
        end = start + t_len

        segment = features[start:end]

        segment_name = f"{case_dir.name}_seg{i}"
        segment_out_dir = output_path / segment_name
        segment_out_dir.mkdir(exist_ok=True)

        np.save(segment_out_dir / "features.npy", segment.astype('float32'))
        shutil.copy(json_file, segment_out_dir / "case.json")

print('Done')

100%|██████████| 25/25 [00:00<00:00, 383.81it/s]

Done


In [10]:
# Segmentation check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/tube_geo/tube_geo_segmented'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0153_seg0: features(20, 3, 66, 65)
case0154_seg0: features(20, 3, 66, 65)
case0155_seg0: features(20, 3, 66, 65)
case0156_seg0: features(20, 3, 66, 65)
case0157_seg0: features(20, 3, 66, 65)
case0158_seg0: features(20, 3, 66, 65)
case0159_seg0: features(20, 3, 66, 65)
case0159_seg1: features(20, 3, 66, 65)
case0162_seg0: features(20, 3, 66, 65)
case0163_seg0: features(20, 3, 66, 65)
case0164_seg0: features(20, 3, 66, 65)
case0164_seg1: features(20, 3, 66, 65)
case0164_seg2: features(20, 3, 66, 65)
case0167_seg0: features(20, 3, 66, 65)
case0168_seg0: features(20, 3, 66, 65)
case0168_seg1: features(20, 3, 66, 65)
case0169_seg0: features(20, 3, 66, 65)
case0169_seg1: features(20, 3, 66, 65)
case0169_seg2: features(20, 3, 66, 65)
case0171_seg0: features(20, 3, 66, 65)
case0172_seg0: features(20, 3, 66, 65)
case0172_seg1: features(20, 3, 66, 65)
case0173_seg0: features(20, 3, 66, 65)
case0173_seg1: features(20, 3, 66, 65)
case0173_seg2: features(20, 3, 66, 65)
case0174_seg0: features(2

In [11]:
# Convert to hdf5

import torch
import torch.nn.functional as F
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_geo/tube_geo_segmented')
output = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_geo/tube_geo_final/tube_geo_converted.h5')

t_len = 20
case_folders = sorted([f for f in input_path.iterdir() if f.is_dir()])
num_segments = len(case_folders)

with h5py.File(output, 'w') as f:
    velocity = f.create_dataset('velocity', shape=(num_segments, t_len, 512, 512, 2),
                                dtype='float32', chunks=(1, t_len, 512, 512, 2))
    particles = f.create_dataset('particles', shape=(num_segments, t_len, 512, 512, 1),
                                 dtype='float32', chunks=(1, t_len, 512, 512, 1))

    for i, case_dir in enumerate(tqdm(case_folders)):
        features = np.load(case_dir / 'features.npy')

        u = features[:, 0, :, :]
        v = features[:, 1, :, :]
        mask = features[:, 2, :, :]

        u_t = torch.from_numpy(u).unsqueeze(1)
        v_t = torch.from_numpy(v).unsqueeze(1)
        mask_t = torch.from_numpy(mask).unsqueeze(1)

        # Upsample to 512x512
        u_up = F.interpolate(u_t, size=(512, 512), mode='bilinear', align_corners=True)
        v_up = F.interpolate(v_t, size=(512, 512), mode='bilinear', align_corners=True)
        mask_up = F.interpolate(mask_t, size=(512, 512), mode='nearest')

        # Permute
        velocity_stack = torch.cat([u_up, v_up], dim=1).permute(0, 2, 3, 1).numpy()
        mask_stack = mask_up.permute(0, 2, 3, 1).numpy()

        velocity[i] = velocity_stack
        particles[i] = mask_stack

print('Done')

100%|██████████| 28/28 [00:03<00:00,  9.28it/s]

Done


In [12]:
# HDF5 check

import h5py

output = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_geo/tube_geo_final/tube_geo_converted.h5')

with h5py.File(output, 'r') as f:
    print(f.keys())
    print(f['velocity'])
    print(f['particles'])

<KeysViewHDF5 ['particles', 'velocity']>
<HDF5 dataset "velocity": shape (28, 20, 512, 512, 2), type "<f4">
<HDF5 dataset "particles": shape (28, 20, 512, 512, 1), type "<f4">


# Inference results for converted TUBE/GEO

In [13]:
# TUBE GEO RESULTS
# INFO - 04/06/26 20:59:58 - 0:04:02 - Evaluation Stats (total size = 28)
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | type         | dim   | size   |   data_loss |   rel l2 |   rel l2 step 1 |   rel l2 step 5 |   rel l2 step 10 |   rel l2 interior |
# +==============+=======+========+=============+==========+=================+=================+==================+===================+
# | incom_ns     | 3     | 28     |    0.916826 |   0.2807 |          0.1904 |          0.2395 |           0.2807 |            0.2756 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | AVE_BY_CLASS | -     | -      |    0.916826 |   0.2807 |          0.1904 |          0.2395 |           0.2807 |            0.2756 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# INFO - 04/06/26 20:59:58 - 0:04:02 - Additional Stats for Rel L2 Error:
#     +----------+--------+--------+--------+--------+--------+----------+
#     | type     |   size |   mean |    std |    min |    max |   median |
#     +==========+========+========+========+========+========+==========+
#     | incom_ns |     28 | 0.2807 | 0.0693 | 0.1934 | 0.4328 |   0.2785 |
#     +----------+--------+--------+--------+--------+--------+----------+
# INFO - 04/06/26 20:59:58 - 0:04:02 - Eval | data loss = 0.916826 | rel l2 = 0.280711 | rel l2 step 1 = 0.190368 | rel l2 step 5 = 0.239509 | rel l2 step 10 = 0.280711 | rel l2 interior = 0.275557
# INFO - 04/06/26 20:59:58 - 0:04:02 -  MEM: 0.00 MB

# Process is repeated for TUBE/BC -> PDEBench INS

In [14]:
# Tube/bc
# Let channel 3 be geo mask
# Remove simulations with <20 timesteps
# For those >20 timesteps, split into segments of 20 and concat accordingly

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/tube/bc'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        u = np.load(os.path.join(case_dir, 'u.npy'), mmap_mode='r')
        v = np.load(os.path.join(case_dir, 'v.npy'), mmap_mode='r')

        print(f"{case}: u{u.shape}, v{v.shape}")

case0000: u(107, 64, 64), v(107, 64, 64)
case0001: u(68, 64, 64), v(68, 64, 64)
case0002: u(47, 64, 64), v(47, 64, 64)
case0003: u(37, 64, 64), v(37, 64, 64)
case0004: u(31, 64, 64), v(31, 64, 64)
case0005: u(27, 64, 64), v(27, 64, 64)
case0006: u(24, 64, 64), v(24, 64, 64)
case0007: u(22, 64, 64), v(22, 64, 64)
case0008: u(21, 64, 64), v(21, 64, 64)
case0009: u(20, 64, 64), v(20, 64, 64)
case0010: u(19, 64, 64), v(19, 64, 64)
case0011: u(18, 64, 64), v(18, 64, 64)
case0012: u(17, 64, 64), v(17, 64, 64)
case0013: u(16, 64, 64), v(16, 64, 64)
case0014: u(16, 64, 64), v(16, 64, 64)
case0015: u(16, 64, 64), v(16, 64, 64)
case0016: u(15, 64, 64), v(15, 64, 64)
case0017: u(14, 64, 64), v(14, 64, 64)
case0018: u(14, 64, 64), v(14, 64, 64)
case0019: u(14, 64, 64), v(14, 64, 64)
case0020: u(15, 64, 64), v(15, 64, 64)
case0021: u(13, 64, 64), v(13, 64, 64)
case0022: u(13, 64, 64), v(13, 64, 64)
case0023: u(13, 64, 64), v(13, 64, 64)
case0024: u(14, 64, 64), v(14, 64, 64)
case0025: u(14, 64, 64)

In [15]:
# Padding
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
import shutil

def load_json(path):
    with open(path, 'r', encoding='utf8') as f:
        return json.load(f)

input_path = Path('/Volumes/T7/CFDBench/tube/bc')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_bc/tube_bc_padded')

for case_dir in tqdm(sorted(input_path.iterdir())):

    u_file = case_dir / "u.npy"
    v_file = case_dir / "v.npy"
    u = np.load(u_file)
    v = np.load(v_file)

    case_params = load_json(case_dir / "case.json")

    mask = np.ones_like(u)

    # Pad left
    u = np.pad(u, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=case_params["vel_in"],)
    v = np.pad(v, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (0, 0), (1, 0)), mode="constant", constant_values=0)
    # # Pad the top and bottom
    u = np.pad(u, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    v = np.pad(v, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    mask = np.pad(mask, ((0, 0), (1, 1), (0, 0)), mode="constant", constant_values=0)
    features = np.stack([u, v, mask], axis=1)  # (T, 3, h, w)

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    np.save(case_out_dir / "features.npy", features)
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')


100%|██████████| 50/50 [00:00<00:00, 142.47it/s]

Done


In [16]:
# Padded check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/tube_bc/tube_bc_padded'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0000: features(107, 3, 66, 65)
case0001: features(68, 3, 66, 65)
case0002: features(47, 3, 66, 65)
case0003: features(37, 3, 66, 65)
case0004: features(31, 3, 66, 65)
case0005: features(27, 3, 66, 65)
case0006: features(24, 3, 66, 65)
case0007: features(22, 3, 66, 65)
case0008: features(21, 3, 66, 65)
case0009: features(20, 3, 66, 65)
case0010: features(19, 3, 66, 65)
case0011: features(18, 3, 66, 65)
case0012: features(17, 3, 66, 65)
case0013: features(16, 3, 66, 65)
case0014: features(16, 3, 66, 65)
case0015: features(16, 3, 66, 65)
case0016: features(15, 3, 66, 65)
case0017: features(14, 3, 66, 65)
case0018: features(14, 3, 66, 65)
case0019: features(14, 3, 66, 65)
case0020: features(15, 3, 66, 65)
case0021: features(13, 3, 66, 65)
case0022: features(13, 3, 66, 65)
case0023: features(13, 3, 66, 65)
case0024: features(14, 3, 66, 65)
case0025: features(14, 3, 66, 65)
case0026: features(14, 3, 66, 65)
case0027: features(13, 3, 66, 65)
case0028: features(13, 3, 66, 65)
case0029: fea

In [17]:
# Convergence
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_bc/tube_bc_padded')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_bc/tube_bc_convergence')

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    converged_idx = num_timesteps - 1

    for i in range(converged_idx):
        u_in, v_in = features[i, 0, :, :], features[i, 1, :, :]
        u_out, v_out = features[i+1, 0, :, :], features[i+1, 1, :, :]

        mag_in = np.sqrt(u_in**2 + v_in**2)
        mag_out = np.sqrt(u_out**2 + v_out**2)

        diff = np.abs(mag_in - mag_out).mean()

        if diff < 1e-3:
            converged_idx = i
            break

    case_out_dir = output_path / case_dir.name
    case_out_dir.mkdir(exist_ok=True)

    truncated_features = features[:converged_idx + 1]
    np.save(case_out_dir / "features.npy", truncated_features.astype('float32'))
    shutil.copy(case_dir / "case.json", case_out_dir / "case.json")

print('Done')

100%|██████████| 50/50 [00:00<00:00, 356.63it/s]

Done


In [18]:
# Convergence check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/tube_bc/tube_bc_convergence'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0000: features(2, 3, 66, 65)
case0001: features(65, 3, 66, 65)
case0002: features(43, 3, 66, 65)
case0003: features(33, 3, 66, 65)
case0004: features(28, 3, 66, 65)
case0005: features(24, 3, 66, 65)
case0006: features(21, 3, 66, 65)
case0007: features(19, 3, 66, 65)
case0008: features(18, 3, 66, 65)
case0009: features(17, 3, 66, 65)
case0010: features(16, 3, 66, 65)
case0011: features(15, 3, 66, 65)
case0012: features(14, 3, 66, 65)
case0013: features(14, 3, 66, 65)
case0014: features(13, 3, 66, 65)
case0015: features(13, 3, 66, 65)
case0016: features(13, 3, 66, 65)
case0017: features(12, 3, 66, 65)
case0018: features(12, 3, 66, 65)
case0019: features(12, 3, 66, 65)
case0020: features(12, 3, 66, 65)
case0021: features(11, 3, 66, 65)
case0022: features(11, 3, 66, 65)
case0023: features(11, 3, 66, 65)
case0024: features(12, 3, 66, 65)
case0025: features(11, 3, 66, 65)
case0026: features(11, 3, 66, 65)
case0027: features(11, 3, 66, 65)
case0028: features(11, 3, 66, 65)
case0029: featu

In [19]:
# Segmentation
import numpy as np
import shutil
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_bc/tube_bc_convergence')
output_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_bc/tube_bc_segmented')

t_len = 20

for case_dir in tqdm(sorted(input_path.iterdir())):

    if not case_dir.is_dir():
        continue

    features_file = case_dir / "features.npy"
    json_file = case_dir / "case.json"

    features = np.load(features_file)
    num_timesteps = features.shape[0]

    num_segments = num_timesteps // t_len

    if num_segments == 0:
        continue

    for i in range(num_segments):
        start = i * t_len
        end = start + t_len

        segment = features[start:end]

        segment_name = f"{case_dir.name}_seg{i}"
        segment_out_dir = output_path / segment_name
        segment_out_dir.mkdir(exist_ok=True)

        np.save(segment_out_dir / "features.npy", segment.astype('float32'))
        shutil.copy(json_file, segment_out_dir / "case.json")

print('Done')

100%|██████████| 50/50 [00:00<00:00, 2349.72it/s]

Done


In [20]:
# Segmentation check

import h5py
import numpy as np
import os

cfd_tube_prop = '/Volumes/T7/CFDBench/Processed_experiment/tube_bc/tube_bc_segmented'

for case in sorted(os.listdir(cfd_tube_prop)):
    case_dir = os.path.join(cfd_tube_prop, case)

    if os.path.isdir(case_dir):
        features = np.load(os.path.join(case_dir, 'features.npy'), mmap_mode='r')

        print(f"{case}: features{features.shape}")

case0001_seg0: features(20, 3, 66, 65)
case0001_seg1: features(20, 3, 66, 65)
case0001_seg2: features(20, 3, 66, 65)
case0002_seg0: features(20, 3, 66, 65)
case0002_seg1: features(20, 3, 66, 65)
case0003_seg0: features(20, 3, 66, 65)
case0004_seg0: features(20, 3, 66, 65)
case0005_seg0: features(20, 3, 66, 65)
case0006_seg0: features(20, 3, 66, 65)


In [21]:
# Convert to hdf5

import torch
import torch.nn.functional as F
import h5py
import numpy as np
from pathlib import Path
from tqdm import tqdm

input_path = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_bc/tube_bc_segmented')
output = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_bc/tube_bc_final/tube_bc_converted.h5')

t_len = 20
case_folders = sorted([f for f in input_path.iterdir() if f.is_dir()])
num_segments = len(case_folders)

with h5py.File(output, 'w') as f:
    velocity = f.create_dataset('velocity', shape=(num_segments, t_len, 512, 512, 2),
                                dtype='float32', chunks=(1, t_len, 512, 512, 2))
    particles = f.create_dataset('particles', shape=(num_segments, t_len, 512, 512, 1),
                                 dtype='float32', chunks=(1, t_len, 512, 512, 1))

    for i, case_dir in enumerate(tqdm(case_folders)):
        features = np.load(case_dir / 'features.npy')

        u = features[:, 0, :, :]
        v = features[:, 1, :, :]
        mask = features[:, 2, :, :]

        u_t = torch.from_numpy(u).unsqueeze(1)
        v_t = torch.from_numpy(v).unsqueeze(1)
        mask_t = torch.from_numpy(mask).unsqueeze(1)

        # Upsample to 512x512
        u_up = F.interpolate(u_t, size=(512, 512), mode='bilinear', align_corners=True)
        v_up = F.interpolate(v_t, size=(512, 512), mode='bilinear', align_corners=True)
        mask_up = F.interpolate(mask_t, size=(512, 512), mode='nearest')

        # Permute
        velocity_stack = torch.cat([u_up, v_up], dim=1).permute(0, 2, 3, 1).numpy()
        mask_stack = mask_up.permute(0, 2, 3, 1).numpy()

        velocity[i] = velocity_stack
        particles[i] = mask_stack

print('Done')

100%|██████████| 9/9 [00:00<00:00,  9.77it/s]

Done


In [22]:
# HDF5 check

import h5py

output = Path('/Volumes/T7/CFDBench/Processed_experiment/tube_bc/tube_bc_final/tube_bc_converted.h5')

with h5py.File(output, 'r') as f:
    print(f.keys())
    print(f['velocity'])
    print(f['particles'])

<KeysViewHDF5 ['particles', 'velocity']>
<HDF5 dataset "velocity": shape (9, 20, 512, 512, 2), type "<f4">
<HDF5 dataset "particles": shape (9, 20, 512, 512, 1), type "<f4">


# Inference results for converted TUBE/BC

In [23]:
# TUBE BC RESULTS
# INFO - 04/06/26 21:13:02 - 0:01:19 - Evaluation Stats (total size = 9)
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | type         | dim   | size   |   data_loss |   rel l2 |   rel l2 step 1 |   rel l2 step 5 |   rel l2 step 10 |   rel l2 interior |
# +==============+=======+========+=============+==========+=================+=================+==================+===================+
# | incom_ns     | 3     | 9      |    0.877274 |   0.2020 |          0.1310 |          0.1710 |           0.2020 |            0.1971 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# | AVE_BY_CLASS | -     | -      |    0.877274 |   0.2020 |          0.1310 |          0.1710 |           0.2020 |            0.1971 |
# +--------------+-------+--------+-------------+----------+-----------------+-----------------+------------------+-------------------+
# INFO - 04/06/26 21:13:02 - 0:01:19 - Additional Stats for Rel L2 Error:
#     +----------+--------+--------+--------+--------+--------+----------+
#     | type     |   size |   mean |    std |    min |    max |   median |
#     +==========+========+========+========+========+========+==========+
#     | incom_ns |      9 | 0.2020 | 0.0822 | 0.1236 | 0.3480 |   0.1592 |
#     +----------+--------+--------+--------+--------+--------+----------+
# INFO - 04/06/26 21:13:02 - 0:01:19 - Eval | data loss = 0.877274 | rel l2 = 0.202046 | rel l2 step 1 = 0.130978 | rel l2 step 5 = 0.170971 | rel l2 step 10 = 0.202046 | rel l2 interior = 0.197136
# INFO - 04/06/26 21:13:02 - 0:01:19 -  MEM: 0.00 MB